# D2 · Four ways to measure importance: impurity, permutation, drop-column and group

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

On a designed dataset (two independent informative features, **a group of three correlated features** carrying the same information, a continuous noise column and a binary one) compare four importance measures of a random forest: impurity, permutation on the test set, **drop-column** (retrain without the feature and see
how much the accuracy drops) and group permutation. Decide which method answers which question.

## Dataset

`generate()` (given): 3000 examples with `x0, x1` independent and informative, `g0, g1, g2` = three noisy versions of the same latent factor `z` (informative), `noise_c` (continuous) and `noise_b` (binary), 60/40 split. Forest: 200 trees, `random_state=0`.

## Educational purpose

Understand that "importance" is not one thing: each method answers a different question and has a blind spot. Know which one to use to decide whether to **remove** a feature, to **explain** the model, to **select** variables.

## Task

1. Build `table` (index: the 7 features; columns `impurity`, `permutation`, `drop_column`) with: impurity importance (`feature_importances_`), permutation importance on the test set (`n_repeats=10`, `random_state=0`, mean, in accuracy units) and **drop-column** (test accuracy of the full model minus that of the model retrained **without that column**, same hyperparameters).
2. Compute `group_permutation`: mean accuracy loss (`n_repeats=10`) permuting `g0, g1, g2` together with the same row permutation, and `group_drop`: accuracy lost retraining without all three.
3. Answer: `impurity_rewards_noise` (True if `noise_c` has an impurity importance above 0.05), `single_permutation_underestimates_group` (True if the sum of the individual permutations of `g0, g1, g2` is less than half of `group_permutation`), `drop_column_underestimates_group` (True if the sum of the drop-columns of the three is less than half of `group_drop`).
4. Save `drop_candidates`: the list of the names of the features with **drop-column ≤ 0.005** (the ones that, looking at them one at a time, seem removable) and `selection_error`: the list of those among `g0, g1, g2` that appear in `drop_candidates` (removing them all together you lose `group_drop` of accuracy).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split


def generate(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    x0, x1, z = rng.normal(size=(3, n))
    g = np.column_stack([z + rng.normal(0, 0.15, n) for _ in range(3)])           # three noisy versions of the same factor
    y = (1.2 * x0 + 0.8 * x1 + 1.5 * z + rng.normal(0, 0.8, n) > 0).astype(int)
    X = np.column_stack([x0, x1, g, rng.normal(size=n), rng.integers(0, 2, n)])
    return X, y


names = ["x0", "x1", "g0", "g1", "g2", "noise_c", "noise_b"]
X, y = generate()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
rf = lambda: RandomForestClassifier(200, random_state=0, n_jobs=-1)
f = rf().fit(X_tr, y_tr)
full_acc = f.score(X_te, y_te)
print("accuracy of the full model:", round(full_acc, 3))

## Your work

In [ ]:
table = pd.DataFrame(index=names, columns=["impurity", "permutation", "drop_column"], dtype=float)
table

In [ ]:
group_permutation = None
group_drop = None

In [ ]:
impurity_rewards_noise = None
single_permutation_underestimates_group = None
drop_column_underestimates_group = None
drop_candidates = None
selection_error = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Drop-column: `A = np.delete(X_tr, j, axis=1); B = np.delete(X_te, j, axis=1); full_acc - rf().fit(A, y_tr).score(B, y_te)`. Group: `np.delete(X_tr, [2, 3, 4], axis=1)`. Group permutation: same row permutation for the three columns (see C2). Permutation method: `permutation_importance(f, X_te, y_te, n_repeats=10, random_state=0).importances_mean`.'):
    check(list(table.index) == names and table.notna().all().all() and close(table["impurity"].sum(), 1.0), "complete table; the impurity importances sum to 1", "seven rows and three columns")
    check(table.loc["noise_c", "impurity"] > 0.05 and table.loc["noise_c", "permutation"] < 0.01 and impurity_rewards_noise is True, "the continuous noise is worth over 5% by impurity and less than 0.01 by permutation", "impurity rewards features with many distinct values")
    check(table.loc[["g0", "g1", "g2"], "drop_column"].max() < 0.01 and table.loc[["g0", "g1", "g2"], "impurity"].min() > 0.1, "each copy has a drop-column below 0.01 (the others replace it) but an impurity importance over 0.1", "the copies are interchangeable")
    check(group_permutation > 0.06 and group_drop > 0.05 and single_permutation_underestimates_group is True and drop_column_underestimates_group is True, "the group together is worth over 0.06 (permutation) and 0.05 (drop); the sum of the single importances is worth less than half of it", "permute or remove the three columns together")
    check("noise_c" in drop_candidates and "noise_b" in drop_candidates and set(selection_error) == {"g0", "g1", "g2"}, "the drop-column alone flags as removable also all three copies of the informative group: selection error", "drop-column <= 0.005")
done()

---
The commented solution is in `solutions/D_interpret/D2_four_importance_methods_compared.ipynb`. Open it after you pass the check, or after you have really tried.